In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [6]:
import kagglehub
path = kagglehub.dataset_download("praveengovi/emotions-dataset-for-nlp")

Using Colab cache for faster access to the 'emotions-dataset-for-nlp' dataset.


In [8]:
df = pd.read_csv(
    path + "/train.txt",
    sep=';',
    header=None,
    names=['text', 'emotion']
)



print(df.head())

                                                text  emotion
0                            i didnt feel humiliated  sadness
1  i can go from feeling so hopeless to so damned...  sadness
2   im grabbing a minute to post i feel greedy wrong    anger
3  i am ever feeling nostalgic about the fireplac...     love
4                               i am feeling grouchy    anger


In [9]:
df.shape

(16000, 2)

In [10]:
df.isnull().sum()

,0
text,0
emotion,0


In [11]:
unique_emotion = df['emotion'].unique()
emotion_number = {}
i = 0
for emo in unique_emotion:
  emotion_number[emo] = i
  i += 1
df['emotion'] = df['emotion'].map(emotion_number)
df

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,5
15998,i feel like this was such a rude comment and i...,1


In [12]:
# Lower text
df['text'] = df['text'].apply(lambda x : x.lower())

In [13]:
# Remove Punctuation
import string

def remove_punc(text):
  return text.translate(str.maketrans('', '', string.punctuation))

df['text'] = df['text'].apply(remove_punc)

In [14]:
# Remove number
def remove_number(text):
  new = ""
  for i in text:
    new = new + i
  return new

df['text'] = df['text'].apply(remove_number)

In [15]:
# Remove Url and Link
import re

def remove_url(text):
  pattern = re.compile(r'https?://\S+|www\.\S+')
  return pattern.sub(r'', text)

df['text'] = df['text'].apply(remove_url)

In [16]:
# Remove HTML tags
import re

def remove_tags(text):
  Pattern = re.compile('<.*?>')
  return Pattern.sub(r'', text)

df['text'] = df['text'].apply(remove_tags)

In [17]:
# Remove Emoji
def remove_emojis(text):
  new = ""
  for i in text:
    if i.isascii():
      new += i
  return new

df['text'] = df['text'].apply(remove_emojis)

In [18]:
# Remove Stopwords
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    word_tokens = word_tokenize(text)
    filtered_text = [word for word in word_tokens if word not in stop_words]
    return " ".join(filtered_text)

df['text'] = df['text'].apply(remove_stopwords)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [19]:
df

,text,emotion
0,didnt feel humiliated,0
1,go feeling hopeless damned hopeful around some...,0
2,im grabbing minute post feel greedy wrong,1
3,ever feeling nostalgic fireplace know still pr...,2
4,feeling grouchy,1
...,...,...
15995,brief time beanbag said anna feel like beaten,0
15996,turning feel pathetic still waiting tables sub...,0
15997,feel strong good overall,5
15998,feel like rude comment im glad,1


In [20]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['emotion'], test_size=0.20, random_state=42)

In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

# TFIDF Vectorizer
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# Bag of Words Vectorizer
bow_vectorizer = CountVectorizer()
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

In [22]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report , accuracy_score
lr_bow = LogisticRegression()
lr_bow.fit(X_train_bow, y_train)

LogisticRegression()

In [23]:
pred_bow = lr_bow.predict(X_test_bow)
print(f'Accuracy: {accuracy_score(y_test, pred_bow)}')
print(f'Test Accuracy: {lr_bow.score(X_test_bow, y_test)}')

Accuracy: 0.88875
Test Accuracy: 0.88875


In [24]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
lr_tfidf = LogisticRegression()
lr_tfidf.fit(X_train_tfidf, y_train)

LogisticRegression()

In [25]:
pred_tfidf = lr_tfidf.predict(X_test_tfidf)
print(f'Accuracy: {accuracy_score(y_test, pred_tfidf)}')
print(f'Test Accuracy: {lr_tfidf.score(X_test_tfidf, y_test)}')

Accuracy: 0.8615625
Test Accuracy: 0.8615625


In [27]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()

param_grid = {
    'C': [0.01, 0.1, 1, 10],
    'max_iter': [100, 300]
}

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    verbose=1,
    n_jobs=-1
)


grid_search.fit(X_train_tfidf, y_train)


print("\nBest Cross Validation Accuracy:")
print(grid_search.best_score_)


best_model = grid_search.best_estimator_

test_accuracy = best_model.score(X_test_tfidf, y_test)

print("\nTest Accuracy:")
print(test_accuracy)

Fitting 5 folds for each of 8 candidates, totalling 40 fits
Best Parameters:
{'C': 10, 'max_iter': 100}

Best Cross Validation Accuracy:
0.8763281249999999

Test Accuracy:
0.8828125


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [28]:
from google.colab import userdata
import os

In [29]:
gem = userdata.get('gemini_key')

In [30]:
os.environ['GEMINI_API_KEY'] = gem

In [31]:
!pip install google-genai

In [32]:
!pip install -U langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 2.1 MB/s eta 0:00:00


In [41]:
from langchain_google_genai import ChatGoogleGenerativeAI


llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite"
)


In [42]:
# ============================================
# STORE GRID SEARCH RESULTS
# ============================================

results_df = pd.DataFrame(grid_search.cv_results_)

# Select Important Columns
results_df = results_df[
    [
        'param_C',
        'param_max_iter',
        'mean_test_score',
        'rank_test_score',
        'mean_fit_time'
    ]
]

print(results_df)

   param_C  param_max_iter  mean_test_score  rank_test_score  mean_fit_time
0     0.01             100         0.358281                7       0.355624
1     0.01             300         0.358281                7       0.317747
2     0.10             100         0.600313                5       0.889548
3     0.10             300         0.600313                5       0.746780
4     1.00             100         0.839609                3       2.161885
5     1.00             300         0.839609                3       2.022249
6    10.00             100         0.876328                1       2.415303
7    10.00             300         0.875391                2       3.076404


In [43]:
tot_prompt = f"""

You are an ML expert.

Analyze the following hyperparameter tuning
results using Tree-of-Thought reasoning.

Tasks:
1. Group promising configurations.
2. Compare bias vs variance.
3. Compare overfitting.
4. Compare training time.
5. Eliminate weak configurations.
6. Select the BEST configuration.

Results:

{results_df.to_string(index=False)}

Provide detailed reasoning.
"""

response = llm.invoke(tot_prompt)

print(response.content)

As an ML expert, I will analyze the provided hyperparameter tuning results using Tree-of-Thought reasoning to systematically evaluate and select the best configuration.

Here's the breakdown of my analysis:

## Tree-of-Thought Reasoning for Hyperparameter Tuning Analysis

**Goal:** Identify the best hyperparameter configuration for the given model and dataset.

**Input:** Hyperparameter tuning results including `param_C`, `param_max_iter`, `mean_test_score`, `rank_test_score`, and `mean_fit_time`.

---

### 1. Group Promising Configurations

**Thought Process:**
The primary indicator of a "promising" configuration is its `mean_test_score`. Higher scores generally mean better performance on unseen data (validation set). I will group configurations based on their performance, prioritizing those with higher `mean_test_score`. Ranks also provide a quick way to identify top performers.

**Steps:**
* Sort the results by `mean_test_score` in descending order.
* Identify the top-ranked configu